# 02 — Explore the live MBTA V3 API

Exploration only: ingestion belongs in `src/`.

**API:** https://api-v3.mbta.com (JSON:API format). It works without a key at 20 requests/minute.
For more headroom, get a free key at https://api-v3.mbta.com and set the `MBTA_API_KEY` environment variable; never paste the key into this notebook.

In [1]:
import json
import os
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the kernel starts in notebooks/
from src.db import database_url  # also loads DATABASE_URL and MBTA_API_KEY from ../.env

import pandas as pd
import psycopg
import requests

BASE_URL = "https://api-v3.mbta.com"
ROUTE = "1"         # API id; the files write it as "01"
FILE_ROUTE = "01"
TODAY = date.today().isoformat()

session = requests.Session()
if os.environ.get("MBTA_API_KEY"):
    session.headers["x-api-key"] = os.environ["MBTA_API_KEY"]


def get(path, **params):
    resp = session.get(f"{BASE_URL}/{path}", params=params, timeout=30)
    resp.raise_for_status()
    print(path, "| rate limit remaining:", resp.headers.get("x-ratelimit-remaining"))
    return resp.json()


def flatten(items):
    """JSON:API resources -> DataFrame of id, attributes and related ids."""
    rows = []
    for item in items:
        row = {"id": item["id"], **item.get("attributes", {})}
        for name, rel in item.get("relationships", {}).items():
            data = rel.get("data")
            if isinstance(data, dict):
                row[f"{name}_id"] = data["id"]
        rows.append(row)
    return pd.DataFrame(rows)

## Bus routes (`route_type` 3)

In [2]:
routes = flatten(get("routes", **{"filter[type]": 3})["data"])
print(len(routes), "bus routes")
routes[["id", "short_name", "long_name", "direction_names", "direction_destinations"]].head(10)

routes | rate limit remaining: 999
150 bus routes


,id,short_name,long_name,direction_names,direction_destinations
0,741,SL1,Logan Airport Terminals - South Station,"[Outbound, Inbound]","[Logan Airport Terminals, South Station]"
1,742,SL2,Design Center - South Station,"[Outbound, Inbound]","[Design Center, South Station]"
2,743,SL3,Chelsea Station - South Station,"[Outbound, Inbound]","[Chelsea Station, South Station]"
3,751,SL4,Nubian Station - South Station,"[Outbound, Inbound]","[Nubian Station, South Station]"
4,749,SL5,Nubian Station - Temple Place,"[Outbound, Inbound]","[Nubian Station, Temple Place]"
5,746,SLW,Silver Line Way - South Station,"[Outbound, Inbound]","[Silver Line Way, South Station]"
6,708,CT3,Beth Israel Deaconess Hospital - Andrew Station,"[Outbound, Inbound]","[Beth Israel Deaconess Hospital, Andrew Station]"
7,1,1,Harvard Square - Nubian Station,"[Outbound, Inbound]","[Harvard Square, Nubian Station]"
8,4,4,North Station - Design Center,"[Outbound, Inbound]","[North Station, Design Center]"
9,7,7,City Point - Otis Street & Summer Street,"[Outbound, Inbound]","[City Point, Otis Street & Summer Street]"


## Route 1: what the API says about it

The files zero-pad single-digit routes (`01`); the API drops the zero, so it's `GET /routes/1`, not `/routes/01`.
`direction_names[i]` names `direction_id = i`.

In [3]:
r1 = get(f"routes/{ROUTE}")["data"]["attributes"]
{k: r1[k] for k in ["short_name", "long_name", "type", "direction_names", "direction_destinations"]}

routes/1 | rate limit remaining: 998


{'short_name': '1',
 'long_name': 'Harvard Square - Nubian Station',
 'type': 3,
 'direction_names': ['Outbound', 'Inbound'],
 'direction_destinations': ['Harvard Square', 'Nubian Station']}

## Schedules: what's *planned* (known in advance, so safe for features)

One day of route 1. First, one raw record, to see what's in `attributes` and what's in `relationships`.

In [4]:
raw = get("schedules", **{"filter[route]": ROUTE, "filter[date]": TODAY})["data"]
print(len(raw), "schedule records for route", ROUTE, "on", TODAY)
print(json.dumps({k: raw[0][k] for k in ["id", "attributes", "relationships"]}, indent=2))

schedules | rate limit remaining: 997
5256 schedule records for route 1 on 2026-10-30
{
  "id": "schedule-78977494-64-1",
  "attributes": {
    "arrival_time": null,
    "departure_time": "2026-10-30T04:37:00-04:00",
    "direction_id": 0,
    "drop_off_type": 1,
    "pickup_type": 0,
    "stop_headsign": null,
    "stop_sequence": 1,
    "timepoint": true
  },
  "relationships": {
    "added_routes": {
      "data": []
    },
    "route": {
      "data": {
        "id": "1",
        "type": "route"
      }
    },
    "stop": {
      "data": {
        "id": "64",
        "type": "stop"
      }
    },
    "trip": {
      "data": {
        "id": "78977494",
        "type": "trip"
      }
    }
  }
}


In [5]:
sched = flatten(raw)
sched["direction"] = sched["direction_id"].map({0: "Outbound", 1: "Inbound"})
print("records:", len(sched))
print("trips:", sched["trip_id"].nunique())
print("distinct stops:", sched["stop_id"].nunique())
print("records per trip:", sched.groupby("trip_id").size().describe()[["min", "50%", "max"]].to_dict())
print("records with no departure_time (last stop):", sched["departure_time"].isna().sum())
sched.groupby("direction").agg(records=("id", "size"), trips=("trip_id", "nunique"))

records: 5256
trips: 219
distinct stops: 46
records per trip: {'min': 24.0, '50%': 24.0, 'max': 24.0}
records with no departure_time (last stop): 219


,records,trips
direction,,
Inbound,2568,107
Outbound,2688,112


In [6]:
sched[["timepoint", "pickup_type", "drop_off_type"]].apply(lambda s: s.value_counts(dropna=False)).T

,False,True
timepoint,5037,219
pickup_type,5037,219
drop_off_type,5037,219


## Predictions: what's expected *right now*

This is real-time data. It's useful for collecting actual departure times (labels), but it's recorded during the trip,
so it must **never** become a feature (leakage).

In [7]:
pred_json = get("predictions", **{"filter[route]": ROUTE, "include": "schedule"})
pred = flatten(pred_json["data"])
print(len(pred), "predictions")
pred.head()

predictions | rate limit remaining: 996
290 predictions


,id,arrival_time,arrival_uncertainty,departure_time,departure_uncertainty,direction_id,last_trip,revenue,schedule_relationship,status,stop_sequence,trip_headsign,update_type,route_id,schedule_id,stop_id,trip_id,vehicle_id
0,prediction-78978566-110-1-1,NaN,NaN,2026-10-30T12:43:00-04:00,NaN,1,False,REVENUE,None,None,1,None,None,1,schedule-78978566-110-1,110,78978566,y1762
1,prediction-78978564-110-1-1,NaN,NaN,2026-10-30T13:02:08-04:00,NaN,1,False,REVENUE,None,None,1,None,None,1,schedule-78978564-110-1,110,78978564,y3245
2,prediction-78978554-64-1-1,NaN,NaN,2026-10-30T12:48:00-04:00,NaN,0,False,REVENUE,None,None,1,None,None,1,schedule-78978554-64-1,64,78978554,y1786
3,prediction-78978550-64-1-1,NaN,NaN,2026-10-30T13:14:00-04:00,NaN,0,False,REVENUE,None,None,1,None,None,1,schedule-78978550-64-1,64,78978550,y1873
4,prediction-78978546-64-1-1,NaN,NaN,2026-10-30T13:27:00-04:00,300.0,0,False,REVENUE,None,None,1,None,None,1,schedule-78978546-64-1,64,78978546,NaN


In [8]:
pred.reindex(columns=["schedule_relationship", "status", "update_type", "revenue"]).apply(lambda s: s.value_counts(dropna=False)).T

,REVENUE,NaN,NaN
schedule_relationship,NaN,NaN,NaN
status,NaN,NaN,NaN
update_type,NaN,NaN,NaN
revenue,290.0,NaN,NaN


### Predicted delay vs. schedule

Join each prediction to its included schedule. This approximates the label the pipeline would compute once the trip is over.
Empty overnight, when no buses are running.

In [9]:
if pred.empty:
    print("No live predictions right now (no buses running?). Re-run during service hours.")
else:
    included = flatten([i for i in pred_json.get("included", []) if i["type"] == "schedule"])
    sched_cols = included[["id", "departure_time", "arrival_time"]].add_prefix("sched_")
    merged = pred.merge(sched_cols, left_on="schedule_id", right_on="sched_id", how="left")
    predicted = pd.to_datetime(merged["departure_time"].fillna(merged["arrival_time"]), utc=True)
    scheduled = pd.to_datetime(merged["sched_departure_time"].fillna(merged["sched_arrival_time"]), utc=True)
    merged["delay_s"] = (predicted - scheduled).dt.total_seconds()
    merged["is_late"] = merged["delay_s"] > 180  # strict >

    print("predictions with no schedule (added trips):", merged["sched_id"].isna().sum())
    print(merged["delay_s"].describe().round(0))
    print("share predicted late:", round(merged.loc[merged["delay_s"].notna(), "is_late"].mean(), 3))

predictions with no schedule (added trips): 0
count    290.0
mean      98.0
std      206.0
min     -310.0
25%      -36.0
50%       36.0
75%      206.0
max      788.0
Name: delay_s, dtype: float64
share predicted late: 0.283


## Vehicles: live positions (real-time, so not a feature)

In [10]:
vehicles = flatten(get("vehicles", **{"filter[route]": ROUTE})["data"])
vehicles.reindex(columns=["id", "label", "current_status", "current_stop_sequence", "occupancy_status", "updated_at"]).head()

vehicles | rate limit remaining: 995


,id,label,current_status,current_stop_sequence,occupancy_status,updated_at
0,y3289,3289,IN_TRANSIT_TO,5,MANY_SEATS_AVAILABLE,2026-10-30T12:36:26-04:00
1,y3245,3245,STOPPED_AT,9,FULL,2026-10-30T12:36:29-04:00
2,y1921,1921,IN_TRANSIT_TO,9,FEW_SEATS_AVAILABLE,2026-10-30T12:36:31-04:00
3,y1875,1875,IN_TRANSIT_TO,2,MANY_SEATS_AVAILABLE,2026-10-30T12:36:29-04:00
4,y1873,1873,IN_TRANSIT_TO,8,FEW_SEATS_AVAILABLE,2026-10-30T12:36:32-04:00


## Alerts: planned detours and service changes

The spec lists alerts as a feature source. Only alerts **announced before** the trip may be used, so filter on `created_at`.

In [11]:
alerts = flatten(get("alerts", **{"filter[route_type]": 3})["data"])
print(len(alerts), "active bus alerts")
alerts.reindex(columns=["effect", "cause", "severity", "created_at", "active_period", "header"]).head()

alerts | rate limit remaining: 994
62 active bus alerts


,effect,cause,severity,created_at,active_period,header
0,DETOUR,SPECIAL_EVENT,7,2026-09-16T13:53:08-04:00,"[{'end': '2026-10-31T03:00:00-04:00', 'start':...","Route 455 outbound is detoured every Friday, S..."
1,STOP_MOVE,CONSTRUCTION,1,2026-06-05T17:07:59-04:00,"[{'end': None, 'start': '2026-06-14T03:00:00-0...",Route 558: Four stops have moved due to constr...
2,DETOUR,SPECIAL_EVENT,7,2026-09-30T15:27:23-04:00,"[{'end': '2026-11-08T13:30:00-05:00', 'start':...",Route 42 is detoured from 7:30 AM to 1:30 PM o...
3,DETOUR,CONSTRUCTION,7,2026-10-30T07:03:06-04:00,"[{'end': '2026-10-30T14:36:30-04:00', 'start':...",Routes 441 and 442 are detoured due to constru...
4,STOP_CLOSURE,CONSTRUCTION,5,2026-10-13T21:45:56-04:00,"[{'end': None, 'start': '2026-10-13T21:45:56-0...",Broadway @ Weston Ave (inbound) is temporarily...


### Which of those stops are timepoints?

The files only hold timepoint stops. Does the API's own `timepoint` flag pick out the same stops as the
historical table, or do we have to filter on the (route, direction, stop) combinations the files contain?

In [12]:
with psycopg.connect(database_url()) as conn:
    hist_stops = pd.DataFrame(
        conn.execute(
            "SELECT DISTINCT direction_id, stop_id FROM stop_events "
            "WHERE source = 'historical' AND route_id = %s", [FILE_ROUTE]
        ).fetchall(),
        columns=["direction", "stop_id"],
    )
    per_day = pd.DataFrame(
        conn.execute(
            "SELECT service_date, count(*) FROM stop_events "
            "WHERE source = 'historical' AND route_id = %s GROUP BY 1", [FILE_ROUTE]
        ).fetchall(),
        columns=["service_date", "rows"],
    )

print("historical timepoints per direction:", hist_stops.groupby("direction").size().to_dict())
in_files = sched.merge(hist_stops, on=["direction", "stop_id"], how="inner")
flagged = sched[sched["timepoint"] == True]
print("API records at a historical timepoint:", len(in_files), "across", in_files["trip_id"].nunique(), "trips")
print("API records flagged timepoint=True:", len(flagged), "| at a historical timepoint:",
      flagged.merge(hist_stops, on=["direction", "stop_id"]).shape[0])
print("stop_sequence of the flagged records:", flagged["stop_sequence"].value_counts().to_dict())

per_day["weekday"] = pd.to_datetime(per_day["service_date"]).dt.day_name()
per_day.groupby("weekday")["rows"].median().rename("historical rows per day, route 01 (median)")

historical timepoints per direction: {'Inbound': 9, 'Outbound': 9}
API records at a historical timepoint: 1971 across 219 trips
API records flagged timepoint=True: 219 | at a historical timepoint: 219
stop_sequence of the flagged records: {1: 219}


weekday
Friday       1971.0
Monday       1935.0
Saturday     2007.0
Sunday       1809.0
Thursday     1935.0
Tuesday      1935.0
Wednesday    1935.0
Name: historical rows per day, route 01 (median), dtype: float64

## How the API compares to the historical CSVs

Run on 2026-10-30 (a Friday), route 1.

### Which API field gives each file column

| File column | API source | Notes |
|---|---|---|
| `service_date` | the `filter[date]` value | not in the record itself |
| `route_id` | `relationships.route.data.id` | files zero-pad (`01`, API `1`); named routes differ (`SL1` = `741`, `CT3` = `708`); look them up by `short_name` in `/routes?filter[type]=3` |
| `direction_id` | `attributes.direction_id` | 0 = Outbound, 1 = Inbound, per `direction_names` on `/routes/1` |
| `stop_id` | `relationships.stop.data.id` | same ids as the files (`64`, `110`, ...) |
| `scheduled` | `attributes.departure_time`, or `arrival_time` at the last stop | ISO 8601 with an Eastern offset (`-04:00`); drop the offset to match the files' wall clock |

### File columns the API has no answer for

- `actual`, `headway`: observed on the day. `/predictions` has live estimates only while a trip runs; the API serves no history.
- `half_trip_id`: different id system. The files use 6-digit half-trip ids (`339926`), the API 8-digit trip ids (`78181030`).
- `time_point_id`, `time_point_order`, `point_type`: the API has `stop_sequence` over *every* stop, not the timepoint order or the timepoint's short name.
- `standard_type`, `scheduled_headway`: not exposed by `/schedules`.

These stay NULL on `source = 'api'` rows in `stop_events`.

### How many records a day

- `/schedules` for route 1 on one day: **5,256 records = 219 trips × 24 stops** (2,688 outbound, 2,568 inbound). 219 records, one per trip, have no `departure_time` (the last stop).
- The files only hold timepoints: 9 per direction on route `01`.
- The API's `timepoint` flag is **not** the same set: it is `true` on 219 records, all at `stop_sequence` 1 (the first stop of each trip).
- Keeping the (route, direction, stop) combinations from the historical table instead leaves **1,971 records = 219 trips × 9 timepoints**, exactly the median Friday for route `01` in the files (1,971). That join is what `src/pull_daily.py` uses.

### Other differences

- The rate limit is 20 requests/minute without a key and 1,000 with one; one `/schedules` request per route is about 150 requests a day.
- `/predictions`, `/vehicles` and real-time `/alerts` are recorded during the trip, so they can never be features; only alerts created before the trip could be.